In [47]:
import pandas as pd
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.model_selection import GridSearchCV

In [48]:
df = pd.read_csv("AgriculturalGeneralEmissions.csv")

In [49]:
new_df = pd.DataFrame(df.values[1:], columns=df.iloc[0])
new_df = new_df.dropna()
new_df = new_df.T
new_df = pd.DataFrame(new_df.values[1:], columns=new_df.iloc[0])
new_df = new_df.reset_index()
new_df

Gas/Source,index,CO2,Urea Fertilization,Liming,CH4,Enteric Fermentation,Manure Management,Rice Cultivation,Field Burning of Agricultural Residues,N2O,Agricultural Soil Management,Manure Management,Field Burning of Agricultural Residues,Total
0,0,7.1,2.4,4.7,241.7,183.1,39.1,18.9,0.5,302.3,288.8,13.4,0.2,551.1
1,1,7.3,2.3,5.0,244.4,183.3,41.0,19.5,0.5,291.5,277.8,13.5,0.2,543.2
2,2,6.9,2.4,4.4,247.7,188.7,39.4,19.0,0.6,288.7,275.0,13.5,0.2,543.2
3,3,6.4,2.6,3.8,252.4,191.4,41.3,19.1,0.6,305.5,292.0,13.4,0.2,564.4
4,4,6.8,2.7,4.1,257.8,194.8,44.3,18.1,0.6,302.9,288.7,14.0,0.2,567.5
5,5,7.0,2.7,4.4,264.5,199.2,45.7,19.1,0.5,302.0,287.5,14.3,0.2,573.6
6,6,6.9,2.6,4.4,263.1,198.0,44.8,19.7,0.6,311.4,296.9,14.3,0.2,581.4
7,7,7.0,2.7,4.3,259.5,194.1,46.9,17.9,0.6,299.6,285.0,14.4,0.2,566.1
8,8,7.7,2.9,4.8,263.8,192.0,50.7,20.4,0.6,305.8,291.1,14.6,0.2,577.3
9,9,7.5,3.0,4.5,262.6,192.2,50.8,19.0,0.6,303.3,288.1,15.1,0.2,573.4


In [59]:
train = new_df.sample(frac=0.7)
test = new_df[~new_df.index.isin(train.index)]
ind = ["index"]
dep = ["CO2"]

In [60]:
pipeline = Pipeline([
    ("poly", PolynomialFeatures()),
    ("scaler", StandardScaler()),
    ("ridge", Ridge())
])

param_grid = {
    "poly__degree": [2,3,4,5],
    "poly__interaction_only": [False, True],
    "ridge__alpha": [0.01,0.1,1,10]
}

grid = GridSearchCV(pipeline, param_grid, cv=5, scoring="r2")
grid.fit(train[ind], train[dep])

best_model = grid.best_estimator_
dep_predict = best_model.predict(test[ind])

print(grid.best_params_)
print("R² Score:", r2_score(test[dep], dep_predict))
print("MSE:", mean_squared_error(test[dep], dep_predict))
print("MAE:", mean_absolute_error(test[dep], dep_predict))

{'poly__degree': 2, 'poly__interaction_only': False, 'ridge__alpha': 1}
R² Score: 0.43185122162284817
MSE: 0.19368191854877098
MAE: 0.36206868405423254
